# Shared Governance at Scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/06_advanced_workflows/shared_governance_scale/shared_governance_scale.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/06_advanced_workflows/shared_governance_scale/shared_governance_scale.ipynb)

## Business Scenario

You are onboarding 500+ tables into Bronze and Silver. You need shared cleansing and validation rules without duplicating YAML across hundreds of contracts.

## Value Proposition

- Policy packs apply shared transformations and row rules
- Base templates standardize metadata, lineage, and dataset rules
- Soft-delete normalization across many tables
- Structured pivot/unpivot transformations keep analytics output consistent
- Registry-driven driver runs hundreds of contracts in parallel

---

## Goals

1. Apply a base template across many contracts
2. Use a shared policy pack for transformations
3. Run parallel ingestion with the driver


## Step 1: Review the Files

Key inputs in this example:

- contracts/_registry.yaml
- contracts/_shared/base_silver.yaml
- contracts/bronze/*.yaml (13 entities)
- contracts/silver/*.yaml (13 entities)
- policy_packs/shared_standard.yaml
- data/*.csv (13 entities)


## Step 2: Apply the Base Template

This deep-merges shared defaults into all Silver contracts, appends shared list rules, and injects
soft-delete handling when operation/deleted_at/is_deleted columns exist.


In [1]:
from pathlib import Path
import subprocess
import sys

def resolve_example_dir(name: str) -> Path:
    cwd = Path.cwd()
    for base in [cwd] + list(cwd.parents):
        if base.name == name and base.exists():
            return base
        candidate = base / name
        if candidate.exists():
            return candidate
        candidate = base / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
        candidate = base / "lakelogic" / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
    return cwd / name

EXAMPLE_DIR = resolve_example_dir("shared_governance_scale")

script_path = EXAMPLE_DIR / "scripts" / "apply_contract_template.py"
if not script_path.exists():
    for parent in EXAMPLE_DIR.parents:
        candidate = parent / "scripts" / "apply_contract_template.py"
        if candidate.exists():
            script_path = candidate
            break

cmd = [
    sys.executable,
    str(script_path),
    "--base-template", str(EXAMPLE_DIR / "contracts" / "_shared" / "base_silver.yaml"),
    "--registry", str(EXAMPLE_DIR / "contracts" / "_registry.yaml"),
    "--stage", "silver",
    "--list-merge-keys", "transformations,quality.row_rules,quality.dataset_rules",
    "--list-mode", "append",
    "--soft-delete",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


C:\Program Files\Python313\python.exe D:\Github\_SaaS\lakelogic\scripts\apply_contract_template.py --base-template D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\_shared\base_silver.yaml --registry D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\_registry.yaml --stage silver --list-merge-keys transformations,quality.row_rules,quality.dataset_rules --list-mode append --soft-delete


CompletedProcess(args=['C:\\Program Files\\Python313\\python.exe', 'D:\\Github\\_SaaS\\lakelogic\\scripts\\apply_contract_template.py', '--base-template', 'D:\\Github\\_SaaS\\lakelogic\\examples\\06_advanced_workflows\\shared_governance_scale\\contracts\\_shared\\base_silver.yaml', '--registry', 'D:\\Github\\_SaaS\\lakelogic\\examples\\06_advanced_workflows\\shared_governance_scale\\contracts\\_registry.yaml', '--stage', 'silver', '--list-merge-keys', 'transformations,quality.row_rules,quality.dataset_rules', '--list-mode', 'append', '--soft-delete'], returncode=0)

## Step 2b: Apply Template via Python (Optional)

Use the same shared template logic without the CLI.


In [2]:
from pathlib import Path

from lakelogic.tools.template_apply import apply_contract_template

def resolve_example_dir(name: str) -> Path:
    cwd = Path.cwd()
    for base in [cwd] + list(cwd.parents):
        if base.name == name and base.exists():
            return base
        candidate = base / name
        if candidate.exists():
            return candidate
        candidate = base / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
        candidate = base / "lakelogic" / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
    return cwd / name

EXAMPLE_DIR = resolve_example_dir("shared_governance_scale")

apply_contract_template(
    base_template=EXAMPLE_DIR / "contracts" / "_shared" / "base_silver.yaml",
    registry=EXAMPLE_DIR / "contracts" / "_registry.yaml",
    stage="silver",
    list_merge_keys=["transformations", "quality.row_rules", "quality.dataset_rules"],
    list_mode="append",
    soft_delete=True,
)


[TemplateApplyResult(contract_path=WindowsPath('D:/Github/_SaaS/lakelogic/examples/06_advanced_workflows/shared_governance_scale/contracts/silver/silver_users.yaml'), output_path=WindowsPath('D:/Github/_SaaS/lakelogic/examples/06_advanced_workflows/shared_governance_scale/contracts/silver/silver_users.yaml'), written=True),
 TemplateApplyResult(contract_path=WindowsPath('D:/Github/_SaaS/lakelogic/examples/06_advanced_workflows/shared_governance_scale/contracts/silver/silver_events.yaml'), output_path=WindowsPath('D:/Github/_SaaS/lakelogic/examples/06_advanced_workflows/shared_governance_scale/contracts/silver/silver_events.yaml'), written=True),
 TemplateApplyResult(contract_path=WindowsPath('D:/Github/_SaaS/lakelogic/examples/06_advanced_workflows/shared_governance_scale/contracts/silver/silver_sessions.yaml'), output_path=WindowsPath('D:/Github/_SaaS/lakelogic/examples/06_advanced_workflows/shared_governance_scale/contracts/silver/silver_sessions.yaml'), written=True),
 TemplateApply

## Step 3: Run in Parallel with a Shared Policy Pack

Policy packs add shared transformations and row rules for every contract (13 entities in this demo).


In [3]:
import shutil
import subprocess
import sys
from pathlib import Path

def resolve_example_dir(name: str) -> Path:
    cwd = Path.cwd()
    for base in [cwd] + list(cwd.parents):
        if base.name == name and base.exists():
            return base
        candidate = base / name
        if candidate.exists():
            return candidate
        candidate = base / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
        candidate = base / "lakelogic" / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
    return cwd / name

EXAMPLE_DIR = resolve_example_dir("shared_governance_scale")

registry_path = EXAMPLE_DIR / "contracts" / "_registry.yaml"
policy_dir = EXAMPLE_DIR / "policy_packs"

def build_driver_cmd(args):
    exe = shutil.which("lakelogic-driver")
    if exe:
        return [exe] + args
    return [sys.executable, "-m", "lakelogic.cli.driver"] + args

cmd = build_driver_cmd([
    "--registry", str(registry_path),
    "--layers", "bronze,silver",
    "--policy-pack", "shared_standard",
    "--policy-pack-dir", str(policy_dir),
    "--max-workers", "8",
])
print(" ".join(cmd))
subprocess.run(cmd, check=True)


C:\Program Files\Python313\python.exe -m lakelogic.cli.driver --registry D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\_registry.yaml --layers bronze,silver --policy-pack shared_standard --policy-pack-dir D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\policy_packs --max-workers 8


CompletedProcess(args=['C:\\Program Files\\Python313\\python.exe', '-m', 'lakelogic.cli.driver', '--registry', 'D:\\Github\\_SaaS\\lakelogic\\examples\\06_advanced_workflows\\shared_governance_scale\\contracts\\_registry.yaml', '--layers', 'bronze,silver', '--policy-pack', 'shared_standard', '--policy-pack-dir', 'D:\\Github\\_SaaS\\lakelogic\\examples\\06_advanced_workflows\\shared_governance_scale\\policy_packs', '--max-workers', '8'], returncode=0)

## Step 3b: Run the Driver in Python (Optional)

This mirrors the CLI invocation using the `PipelineDriver` API.


In [4]:
from pathlib import Path

from lakelogic.cli.driver import PipelineDriver, Window

def resolve_example_dir(name: str) -> Path:
    cwd = Path.cwd()
    for base in [cwd] + list(cwd.parents):
        if base.name == name and base.exists():
            return base
        candidate = base / name
        if candidate.exists():
            return candidate
        candidate = base / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
        candidate = base / "lakelogic" / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
    return cwd / name

EXAMPLE_DIR = resolve_example_dir("shared_governance_scale")

registry_paths = {
    "system": EXAMPLE_DIR / "contracts" / "_registry.yaml",
    "reference": None,
    "gold": None,
}

driver = PipelineDriver(
    engine="polars",
    max_workers=8,
    policy_pack="shared_standard",
    policy_pack_dir=EXAMPLE_DIR / "policy_packs",
)

driver.run(
    registry_paths,
    layers=["bronze", "silver"],
    window=Window(None, None, "full"),
    reprocess=False,
)


2026-02-15 12:24:36.484 | INFO     | lakelogic.core.processor:run_source:408 - Loading source: D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\bronze\..\..\data\events.csv via polars


2026-02-15 12:24:36.489 | INFO     | lakelogic.core.processor:run_source:408 - Loading source: D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\bronze\..\..\data\users.csv via polars


2026-02-15 12:24:36.489 | INFO     | lakelogic.core.processor:run_source:408 - Loading source: D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\bronze\..\..\data\logins.csv via polars


2026-02-15 12:24:36.489 | INFO     | lakelogic.core.processor:run_source:408 - Loading source: D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\bronze\..\..\data\sessions.csv via polars


2026-02-15 12:24:36.517 | INFO     | lakelogic.core.processor:run_source:408 - Loading source: D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\bronze\..\..\data\purchases.csv via polars


...

2026-02-15 12:24:40.151 | INFO     | lakelogic.core.materialization:materialize_dataframe:987 - Materialized 72 rows to D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\silver\..\..\output\silver\daily_metrics\data.parquet


2026-02-15 12:24:40.152 | INFO     | lakelogic.cli.driver:_run_contract:580 - silver_daily_metrics: full load executed


2026-02-15 12:24:40.160 | INFO     | lakelogic.core.processor:run:319 - Run complete. [domain=growth, system=app_events] Source: 3, Total (post-transform): 0, Good: 0, Quarantined: 0, Pre-Transform Dropped: 3, Ratio: n/a


2026-02-15 12:24:40.185 | INFO     | lakelogic.core.materialization:materialize_dataframe:987 - Materialized 0 rows to D:\Github\_SaaS\lakelogic\examples\06_advanced_workflows\shared_governance_scale\contracts\silver\..\..\output\silver\clicks\data.parquet


2026-02-15 12:24:40.187 | INFO     | lakelogic.cli.driver:_run_contract:580 - silver_clicks: full load executed


## Step 4: Structured Pivot + Unpivot

- `silver_feature_flags.yaml` uses `pivot` to create per-user feature flag columns.
- `silver_daily_metrics.yaml` uses `unpivot` to normalize wide metrics into long format.


## Step 5: Inspect Outputs

After running, load the output Parquet files using a LakeLogic engine.


In [5]:
from pathlib import Path

from lakelogic import DataProcessor

def resolve_example_dir(name: str) -> Path:
    cwd = Path.cwd()
    for base in [cwd] + list(cwd.parents):
        if base.name == name and base.exists():
            return base
        candidate = base / name
        if candidate.exists():
            return candidate
        candidate = base / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
        candidate = base / "lakelogic" / "examples" / "06_advanced_workflows" / name
        if candidate.exists():
            return candidate
    return cwd / name

EXAMPLE_DIR = resolve_example_dir("shared_governance_scale")

OUTPUT_DIR = EXAMPLE_DIR / "output" / "silver"

def load_silver(name: str):
    contract = {
        "info": {"title": f"Inspect {name}", "version": "1.0.0"},
        "dataset": f"inspect_{name}",
        "source": {"type": "landing", "path": str(OUTPUT_DIR / name / "*.parquet"), "load_mode": "full"},
        "quality": {"enforce_required": False},
    }
    processor = DataProcessor(contract)
    result = processor.run_source()
    return result.good

